# Fuel Lattice Parameter: Model Creation and Cross-Validation

Trains the five candidate models (Linear Regression, Lumped RF, Independent RF,
Lumped GBR, Independent GBR) on the full filtered dataset, and separately runs
5-fold cross-validation to produce held-out accuracy metrics.

**Every hyperparameter lives in exactly one place: `engine/train_models.py`'s
`model_estimators()`.** This notebook never constructs an estimator directly, but calls
into the engine, the same code path `cli.py train` and `cli.py evaluate` use. If you
change a hyperparameter here without changing it in `engine/`, `tests/test_params_parity.py`
will fail, which is intentional: it is the guard that keeps this notebook and the
shipped model in sync.

Only `rf1` (Lumped RF) is committed as a trained binary in this repository
(`Models/binaries/LumpedRFModel.joblib`). Training the others here is useful for
experimentation, but re-running this notebook does **not** change what `cli.py predict`
uses unless you explicitly overwrite `Models/binaries/LumpedRFModel.joblib`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from engine import config, artifacts
from engine.train_models import prepare_training_frame, model_estimators, train
from engine.evaluate_cv import evaluate
from engine.reporting import plot_actual_vs_predicted

## Prepare the training frame

Loads the featurized dataset, applies the dedup/filter/feature-selection pipeline, and
saves `Dataset/Training_Dataset.csv` + `Models/feature_labels/ML_FeatureLabels.joblib`
(+ its JSON mirror). This is the exact frame every model below is fit on.

In [ ]:
df, X, Y, feature_labels = prepare_training_frame()
print(f"Training frame: {len(df)} rows, {len(feature_labels)} features")
display(df.head())

## Full-fit training

Fits every model in `model_estimators()` on the complete training frame (not a
train/test split, since that split happens only inside the cross-validation step below) and
saves each to `Models/binaries/`.

This can take a while for `gbr1`/`gbr2` (1,800 boosting rounds each). Pass a subset of
keys to `train()` to skip models you don't need, e.g. `train(["rf1"])` for just the
headline model.

In [ ]:
trained = train(list(config.MODEL_FILES.keys()))
for key, path in trained.items():
    print(f"{key}: {config.MODEL_FILES[key][1]} -> {path}")

## Cross-validation metrics

5-fold `KFold` cross-validation via `engine.evaluate_cv.evaluate()`. This refits fresh
estimators per fold, and deliberately never touches the saved binaries above, because
CV metrics describe the hyperparameters and the training frame, not one particular
fitted model. Results are written to `Results/metrics/ModelMetrics_CrossVal.csv`, which
is what `Models/LumpedRFModel/model_card.md` quotes.

Excludes Linear Regression (`lin`) by default, as set by `config.REPORTABLE`. LR is a
baseline sanity check only and is never reported to end users.

In [ ]:
metrics_df = evaluate(keys=config.REPORTABLE, out_dir=str(config.FIGURES_DIR))
display(metrics_df)

## Actual-vs-predicted plots

Generates the 3-panel actual-vs-predicted grid for each evaluated model, colored by
crystal system, and saves each to `Results/figures/`. This is the only place any figure
in this repository gets written to disk, and no notebook calls `savefig()` directly.

In [ ]:
from engine import config as _config  # noqa: F401 (re-import for clarity in this cell)

crystal_systems = [lbl.removeprefix("cs_") for lbl in _config.CS_LABELS]

for key in config.REPORTABLE:
    est = model_estimators()[key]
    est.fit(X, Y)
    Y_pred = pd.DataFrame(est.predict(X), columns=[f"{p}_pred" for p in ["a", "b", "c"]])

    model_name = config.MODEL_FILES[key][1]
    row = metrics_df[metrics_df["model_key"] == key]
    mse_vals = row.set_index("param").reindex(["a", "b", "c"])["MSE_all"].values
    mae_vals = row.set_index("param").reindex(["a", "b", "c"])["MAE_all"].values
    r2_vals  = row.set_index("param").reindex(["a", "b", "c"])["R2_all"].values

    out_path = plot_actual_vs_predicted(
        Y, Y_pred, model_name,
        mse_vals=mse_vals, mae_vals=mae_vals, r2_vals=r2_vals,
        df=df, col="crystal_system", labels=crystal_systems,
        outdir=config.FIGURES_DIR,
    )
    print(f"{model_name}: {out_path}")